# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huydang2006/flyrank-ML-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane 4: CTR / Engagement Opportunity Scoring.** 

I chose this lane because it directly addresses a concrete, focused decision problem: which visible pages are under-capturing clicks within their search position tier?

Even though it outperforms most `deep` pages, a page ranked in `page_1` with a 0.01% CTR is underperforming compared to other `page_1` pages, *which typically get a 0.23% CTR*. This tier-adjusted approach catches real opportunities that position-blind blending would miss, and it guides actionable decisions: rewrite title/meta, improve intent match, improve engagement, or monitor. This is decision-support: I produce a ranked list with reason codes so reviewers know what to fix.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**The decision:** Which visible pages are under-capturing clicks relative to others in the same search position tier, and are worth review for content or metadata improvement?

**Who acts on it:** Content reviewers, SEO strategists, or editors who have a fixed budget each week to audit and improve pages. They need a ranked list with reason codes: top 100 pages flagged as "high impressions + low CTR for tier" or "strong position + weak engagement."

**What does a wrong call cost?** Two errors hurt:
- **False positive (flagging a low-volume page):** Reviewer wastes time auditing a page with only 100 trailing impressions. Even if CTR is low, the volume is noise. Wasted effort.
- **False negative (missing a high-potential page):** A page with ~5,000 impressions at `page_1` has 0.01% CTR, but `page_1` pages typically earn 0.23% CTR. The gap suggests a fixable title/meta problem. If we miss it, 5,000 impressions stay uncaptured — that's real opportunity cost.

**Why a plain rule isn't enough:** Ranking all pages by position alone misses tier-specific underperformers. Ranking by impressions alone ignores position context (a 1,000-impression page at `deep` is different from `page_1`). A simple if-statement like "flag pages with CTR < 0.1%" catches noise and misses context. I need to calculate expected CTR *by tier*, then flag pages that sit far below their tier's median. That's the signal.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [2]:
# Setup for running in Colab or local environment
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # For local execution, navigate to repo root
    import pathlib
    notebook_path = pathlib.Path.cwd()
    while notebook_path != notebook_path.parent:
        if (notebook_path / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            os.chdir(notebook_path)
            break
        notebook_path = notebook_path.parent

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f'Dataset: {len(df):,} pages across {df["client_id"].nunique()} clients')

Dataset: 30,000 pages across 32 clients


In [11]:
# 1. Expected CTR by position tier
print('Expected CTR by position tier')
print('(This is the benchmark for each tier)')
print()

position_tiers = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
has_position_data = df[(df['impressions_90d'] >= 100) & (df['avg_position'] > 0)].copy()

tier_benchmarks = {}
for tier in position_tiers:
    tier_data = has_position_data[has_position_data['position_tier'] == tier]
    if len(tier_data) > 0:
        expected_ctr = tier_data['ctr'].median()
        tier_benchmarks[tier] = expected_ctr
        print(f'  {tier:12s}: expected CTR = {expected_ctr:5.2f}%  (n={len(tier_data):5,} pages)')

# 2. Pages with big CTR gaps below their tier benchmark
print()
print('Pages with big CTR gaps below their tier benchmark')
print('(These are candidates for metadata or content review)')
print()

# Add tier benchmark and gap columns
has_position_data['expected_ctr'] = has_position_data['position_tier'].map(tier_benchmarks)
has_position_data['ctr_gap'] = has_position_data['ctr'] - has_position_data['expected_ctr']
# Relative gap: how far below expected as a percentage (avoids tier bias)
has_position_data['relative_ctr_gap'] = np.where(
    has_position_data['expected_ctr'] > 0,
    (has_position_data['ctr'] - has_position_data['expected_ctr']) / has_position_data['expected_ctr'],
    0.0  # If expected CTR is 0 (deep tier), treat as no gap
)

# Filter for sufficient volume (>= 100 impressions) to avoid noise
# Use relative gap < -0.5 (at least 50% below expected CTR for the tier)
candidates = has_position_data[
    (has_position_data['impressions_90d'] >= 500) &
    (has_position_data['relative_ctr_gap'] < -0.5)
].copy()

# 3. Sort by relative gap (worst underperformers first), then by impressions for ties
candidates = candidates.sort_values(['relative_ctr_gap', 'impressions_90d'], ascending=[True, False])

print(f'  Total candidate pages (volume >= 500 impr, relative gap < -50%): {len(candidates):,}')
print()
print('  Top 10 biggest underperformers:')
print()

for idx, (i, row) in enumerate(candidates.head(10).iterrows(), 1):
    reason_codes = []
    if row['impressions_90d'] >= 5000:
        reason_codes.append('high_impressions')
    if row['ctr'] < (tier_benchmarks[row['position_tier']] * 0.5):
        reason_codes.append('very_low_ctr')
    if row['engagement_rate'] < 30.0:
        reason_codes.append('weak_engagement')
    
    print(f'  {idx}. {row["position_tier"]:12s} | Impr={row["impressions_90d"]:6.0f} | CTR={row["ctr"]:5.2f}% vs {row["expected_ctr"]:5.2f}% (rel gap={row["relative_ctr_gap"]*100:6.1f}%)')
    print(f'     Reasons: {", ".join(reason_codes) if reason_codes else "low_ctr_for_tier"}')
    print()

print()
print('Tier distribution of candidates')
print('(Sanity check: should see all tiers, not just top positions)')
print()

tier_distribution = candidates['position_tier'].value_counts()
for tier in position_tiers:
    count = tier_distribution.get(tier, 0)
    pct = (count / len(candidates) * 100) if len(candidates) > 0 else 0
    print(f'  {tier:12s}: {count:5,} pages ({pct:5.1f}%)')

Expected CTR by position tier
(This is the benchmark for each tier)

  top_3       : expected CTR =  0.19%  (n=  533 pages)
  page_1      : expected CTR =  0.23%  (n=8,633 pages)
  striking    : expected CTR =  0.15%  (n=5,903 pages)
  page_3_5    : expected CTR =  0.06%  (n=6,058 pages)
  deep        : expected CTR =  0.00%  (n=  879 pages)

Pages with big CTR gaps below their tier benchmark
(These are candidates for metadata or content review)

  Total candidate pages (volume >= 500 impr, relative gap < -50%): 4,125

  Top 10 biggest underperformers:

  1. page_1       | Impr=208678 | CTR= 0.00% vs  0.23% (rel gap=-100.0%)
     Reasons: high_impressions, very_low_ctr, weak_engagement

  2. page_3_5     | Impr= 84093 | CTR= 0.00% vs  0.06% (rel gap=-100.0%)
     Reasons: high_impressions, very_low_ctr, weak_engagement

  3. page_3_5     | Impr= 41226 | CTR= 0.00% vs  0.06% (rel gap=-100.0%)
     Reasons: high_impressions, very_low_ctr, weak_engagement

  4. page_3_5     | Impr= 32491 

In [12]:
# Baseline comparison

# Baseline 1: rank by impressions only (ignores position context)
baseline_impressions = has_position_data.sort_values('impressions_90d', ascending=False).head(100)
# Baseline 2: rank by raw CTR only (ignores position tier)
baseline_ctr = has_position_data[
    (has_position_data['impressions_90d'] >= 100)
].sort_values('ctr', ascending=False).head(100)
# Tier-adjusted candidates (already computed above)
tier_adjusted = candidates.head(100)

# Compare overlap
baseline_imp_ids = set(baseline_impressions['content_id'])
baseline_ctr_ids = set(baseline_ctr['content_id'])
tier_adjusted_ids = set(tier_adjusted['content_id'])

print(f'Baseline (impressions-only) overlap with baseline (CTR-only): {len(baseline_imp_ids & baseline_ctr_ids)} / 100')
print(f'Baseline (impressions-only) overlap with tier-adjusted: {len(baseline_imp_ids & tier_adjusted_ids)} / 100')
print(f'Baseline (CTR-only) overlap with tier-adjusted: {len(baseline_ctr_ids & tier_adjusted_ids)} / 100')
print(f'Tier-adjusted unique candidates (not in either baseline): {len(tier_adjusted_ids - baseline_imp_ids - baseline_ctr_ids)}')

print(f'\nTier diversity in top 100:')
print(f'  Tier-adjusted: {tier_adjusted["position_tier"].nunique()} tiers')
print(f'  Baseline (impressions): {baseline_impressions["position_tier"].nunique()} tiers')
print(f'  Baseline (CTR): {baseline_ctr["position_tier"].nunique()} tiers')
print()

# Quantify false positive and false negative costs
low_volume_flagged = candidates[candidates['impressions_90d'] < 1000]
print(f'False positive risk (flagged but <1000 impressions): {len(low_volume_flagged)} pages')
print(f'  Average impressions: {low_volume_flagged["impressions_90d"].mean():.0f}')

high_volume_gaps = has_position_data[
    (has_position_data['impressions_90d'] >= 5000) &
    (has_position_data['relative_ctr_gap'] < -0.5)
]
print(f'\nFalse negative risk (high volume >=5000 + relative gap < -50%): {len(high_volume_gaps)} pages')
potential_uncaptured = (high_volume_gaps['impressions_90d'] * high_volume_gaps['ctr_gap'].abs() / 100).sum()
print(f'  Potential uncaptured clicks: {potential_uncaptured:.0f}')

print(f'\nCandidate volume distribution:')
print(f'  < 1000 impressions:  {len(candidates[candidates["impressions_90d"] < 1000]):,} pages')
print(f'  1000-4999 impr:     {len(candidates[(candidates["impressions_90d"] >= 1000) & (candidates["impressions_90d"] < 5000)]):,} pages')
print(f'  >= 5000 impressions: {len(candidates[candidates["impressions_90d"] >= 5000]):,} pages')

Baseline (impressions-only) overlap with baseline (CTR-only): 0 / 100
Baseline (impressions-only) overlap with tier-adjusted: 1 / 100
Baseline (CTR-only) overlap with tier-adjusted: 0 / 100
Tier-adjusted unique candidates (not in either baseline): 99

Tier diversity in top 100:
  Tier-adjusted: 4 tiers
  Baseline (impressions): 5 tiers
  Baseline (CTR): 4 tiers

False positive risk (flagged but <1000 impressions): 1281 pages
  Average impressions: 716

False negative risk (high volume >=5000 + relative gap < -50%): 956 pages
  Potential uncaptured clicks: 28198

Candidate volume distribution:
  < 1000 impressions:  1,281 pages
  1000-4999 impr:     1,888 pages
  >= 5000 impressions: 956 pages


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I CAN claim

- **Observed:** I can observe and measure CTR, position, impressions, clicks, and engagement rate directly in the historical data. No assumptions needed.
- **Directional:** I can show that pages with low CTR relative to their position tier have historically underperformed. If a page improves its CTR within its tier, I can show the directional impact on clicks.
- **Decision-support:** I can rank pages by CTR gap within tier and flag them with reason codes (high volume + low CTR, weak engagement, etc.). Reviewers can use this ranked list to allocate audit effort more efficiently than position-only ranking.
- **Baseline comparison:** I can build a simple baseline (e.g., "rank by position only") and show that my tier-adjusted score produces a different, more diverse set of candidates.

### What I CANNOT claim

- **Causation / "If a reviewer fixes this page's title, CTR will improve by X%."** I cannot prove what a reviewer's action will cause. I observe correlation in historical data, not cause and effect. Content quality, user intent changes, and algorithm updates are confounded.
- **Predicting Google's ranking changes.** I cannot predict whether Google will maintain or improve a page's position after a reviewer updates it.
- **Predicting absolute future CTR.** Search behavior, competition, and Google's algorithm change. A 0.76% CTR today does not guarantee 0.76% next month.
- **Complete strategies.** This score is decision-support for content review prioritization, not a complete strategy. Off-page signals, competitive landscape, and editorial judgment matter too.

### My validation (sanity checks on the ranked list)

I will validate by checking:
1. **Volume floor:** My top-100 candidates have ≥ 100 impressions (no noise)
2. **Tier diversity:** My candidates span all five tiers (not just top_3), proving tier adjustment matters
3. **Reason codes plausibility:** Can I verify each reason code (high_impressions ≥ 1000?, very_low_ctr < 50% of tier median?) against the data?
4. **Gap consistency:** Pages I rank higher have larger CTR gaps within their tier
5. **Baseline comparison:** My tier-adjusted ranking produces a different top-100 than position-only ranking (if it's identical, I haven't added value)

I will report honestly: if my ranking doesn't beat the baseline, I'll say so.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.